# Cyberbullying Detection — Colab Training Notebook

**Run every cell from top to bottom in order.**

## Before you start
1. Go to **Runtime → Change runtime type → T4 GPU** → Save
2. Compress your local project folder (`cyberbulling-detection`) into a ZIP file
3. Come back here and run the cells below

## What this notebook does
| Step | Action |
|---|---|
| 1 | Verify GPU is available |
| 2 | Mount Google Drive (to persist checkpoints) |
| 3 | Upload & extract your project ZIP |
| 4 | Install dependencies |
| 5 | Fix dataset (drop null row, create placeholder image) |
| 6 | Train with **m-BERT** |
| 7 | Evaluate m-BERT model |
| 8 | Train with **MuRIL** |
| 9 | Evaluate MuRIL model |
| 10 | Download model checkpoints to your computer |

---
## Step 1 — Verify GPU

You should see a Tesla T4 (or similar). If you see `No GPU`, go to **Runtime → Change runtime type → GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✅ GPU found: {gpu}  ({mem:.1f} GB VRAM)')
else:
    print('❌ No GPU detected. Go to Runtime → Change runtime type → GPU and re-run.')
    raise SystemExit('GPU required')

---
## Step 2 — Mount Google Drive

This saves your trained model checkpoints to Google Drive so they survive if the Colab session disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
GDRIVE_SAVE_DIR = '/content/drive/MyDrive/cyberbullying_models'
os.makedirs(GDRIVE_SAVE_DIR, exist_ok=True)
print(f'Model checkpoints will be saved to: {GDRIVE_SAVE_DIR}')

---
## Step 3 — Upload & Extract Project ZIP

Run this cell. A file picker will appear — select `cyberbulling-detection.zip` from your computer.

In [ ]:
from google.colab import files
import zipfile, os, shutil

print('Select your cyberbulling-detection.zip file...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

# Extract to /content/
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Find the extracted project root (handles varied zip structures)
candidates = [d for d in os.listdir('/content/')
              if os.path.isdir(f'/content/{d}') and 'cyberbull' in d.lower()]

if not candidates:
    # Files may have extracted directly into /content/
    PROJECT_DIR = '/content'
else:
    PROJECT_DIR = f'/content/{candidates[0]}'

print(f'Project directory: {PROJECT_DIR}')
print('Contents:', os.listdir(PROJECT_DIR))

In [ ]:
# Change working directory to project root
import os, sys
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
print('Working directory:', os.getcwd())

---
## Step 4 — Install Dependencies

Colab already has PyTorch and torchvision installed. We only need `transformers` and any missing packages.

In [ ]:
!pip install -q transformers==4.40.0 accelerate -U
print('\n✅ Dependencies installed')

---
## Step 5 — Fix Dataset

Two quick fixes before training:
1. Drop the 1 row with a null `text_content`
2. Create the `data/images/none.jpg` placeholder (all rows reference this file)

In [ ]:
import pandas as pd
from PIL import Image
import os

# Fix 1: Drop null row and save clean CSV
raw_csv   = 'data/train.csv'
clean_csv = 'data/train_clean.csv'
df = pd.read_csv(raw_csv)
df_clean = df.dropna(subset=['text_content']).reset_index(drop=True)
df_clean.to_csv(clean_csv, index=False)
print(f'Original rows : {len(df):,}')
print(f'After cleaning: {len(df_clean):,} (dropped {len(df)-len(df_clean)} null row)')

# Fix 2: Create placeholder image
os.makedirs('data/images', exist_ok=True)
img_path = 'data/images/none.jpg'
if not os.path.exists(img_path):
    Image.new('RGB', (224, 224), color=(128, 128, 128)).save(img_path)
print(f'\nImage placeholder: {img_path} ✅')

# Verify label distribution
print('\nLabel distribution:')
for col in ['aggression', 'repetition', 'intent']:
    pos = int(df_clean[col].sum())
    total = len(df_clean)
    print(f'  {col:12s}: {pos:4d}/{total} positive ({pos/total*100:.1f}%)')

In [ ]:
# Point config to the clean CSV (in-memory patch — no file edit needed)
import src.config as cfg
cfg.TRAIN_CSV = os.path.join(cfg.DATA_DIR, 'train_clean.csv')
print(f'TRAIN_CSV set to: {cfg.TRAIN_CSV}')
print(f'IMAGE_DIR set to: {cfg.IMAGE_DIR}')
print(f'DEVICE         : {cfg.DEVICE}')
print(f'TEXT_MODEL     : {cfg.TEXT_MODEL}')
print(f'EPOCHS         : {cfg.EPOCHS}')
print(f'BATCH_SIZE     : {cfg.BATCH_SIZE}')

---
## Step 6 — Train with m-BERT

**Expected time on T4 GPU: ~3–8 minutes per epoch** (10 epochs max, early stopping may stop earlier).

Watch for:
- `Val Loss` decreasing each epoch — good
- `✓ Best model saved` — checkpoint written
- `Early stopping` — training stopped because val loss stopped improving (this is fine)

In [ ]:
# Reload modules to pick up config changes
import importlib
import src.train as train_module
importlib.reload(train_module)

print('=' * 60)
print('  TRAINING — m-BERT (bert-base-multilingual-cased)')
print('=' * 60)

mbert_model = train_module.train(
    csv_path=cfg.TRAIN_CSV,
    image_dir=cfg.IMAGE_DIR,
    text_model='bert-base-multilingual-cased'
)

In [ ]:
# Backup m-BERT checkpoints to Google Drive immediately
import shutil, os

mbert_dir = os.path.join(GDRIVE_SAVE_DIR, 'mbert')
os.makedirs(mbert_dir, exist_ok=True)

for fname in ['best_model.pth', 'final_model.pth']:
    src_path = os.path.join('models', fname)
    dst_path = os.path.join(mbert_dir, fname)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size_mb = os.path.getsize(dst_path) / 1e6
        print(f'Backed up: {dst_path}  ({size_mb:.1f} MB)')
    else:
        print(f'Not found: {src_path}')

---
## Step 7 — Evaluate m-BERT Model

In [ ]:
import src.evaluate as eval_module
importlib.reload(eval_module)

print('=' * 60)
print('  EVALUATION — m-BERT')
print('=' * 60)

eval_module.evaluate(
    model_path=os.path.join('models', 'best_model.pth'),
    csv_path=cfg.TRAIN_CSV
)

In [ ]:
# Backup evaluation report to Drive
report_src = os.path.join('models', 'evaluation_report.txt')
report_dst = os.path.join(mbert_dir, 'evaluation_report.txt')
if os.path.exists(report_src):
    shutil.copy2(report_src, report_dst)
    print(f'Report saved: {report_dst}')
    # Print report content
    with open(report_src) as f:
        print(f.read())

---
## Step 8 — Train with MuRIL

MuRIL (`google/muril-base-cased`) was pre-trained on 17 Indian languages plus their transliterated forms.
It natively understands Roman Urdu text better than m-BERT, so it is expected to outperform m-BERT on this dataset.

**Note:** MuRIL (~900 MB) will be downloaded automatically on first use.

In [ ]:
# Rename m-BERT checkpoints before MuRIL training overwrites them
import shutil
for fname in ['best_model.pth', 'final_model.pth']:
    src = os.path.join('models', fname)
    dst = os.path.join('models', fname.replace('.pth', '_mbert.pth'))
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Renamed: {fname} → {os.path.basename(dst)}')

In [ ]:
importlib.reload(train_module)

print('=' * 60)
print('  TRAINING — MuRIL (google/muril-base-cased)')
print('=' * 60)

muril_model = train_module.train(
    csv_path=cfg.TRAIN_CSV,
    image_dir=cfg.IMAGE_DIR,
    text_model='google/muril-base-cased'
)

In [ ]:
# Backup MuRIL checkpoints to Google Drive
muril_dir = os.path.join(GDRIVE_SAVE_DIR, 'muril')
os.makedirs(muril_dir, exist_ok=True)

for fname in ['best_model.pth', 'final_model.pth']:
    src_path = os.path.join('models', fname)
    dst_path = os.path.join(muril_dir, fname)
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        size_mb = os.path.getsize(dst_path) / 1e6
        print(f'Backed up: {dst_path}  ({size_mb:.1f} MB)')

---
## Step 9 — Evaluate MuRIL Model

In [ ]:
# Switch config to MuRIL so evaluate.py uses the right tokenizer
cfg.TEXT_MODEL = 'google/muril-base-cased'

importlib.reload(eval_module)

print('=' * 60)
print('  EVALUATION — MuRIL')
print('=' * 60)

eval_module.evaluate(
    model_path=os.path.join('models', 'best_model.pth'),
    csv_path=cfg.TRAIN_CSV
)

In [ ]:
# Backup MuRIL evaluation report
report_dst = os.path.join(muril_dir, 'evaluation_report.txt')
if os.path.exists(report_src):
    shutil.copy2(report_src, report_dst)
    print(f'Report saved: {report_dst}')
    with open(report_src) as f:
        print(f.read())

---
## Step 10 — Download Model Checkpoints

These files will download to your computer's default Downloads folder.
Place them in the `models/` directory of your local project.

In [ ]:
from google.colab import files

to_download = [
    ('models/best_model_mbert.pth',  'best_model_mbert.pth'),
    ('models/final_model_mbert.pth', 'final_model_mbert.pth'),
    ('models/best_model.pth',        'best_model_muril.pth'),
    ('models/final_model.pth',       'final_model_muril.pth'),
]

for src_path, display_name in to_download:
    if os.path.exists(src_path):
        size_mb = os.path.getsize(src_path) / 1e6
        print(f'Downloading {display_name} ({size_mb:.1f} MB)...')
        files.download(src_path)
    else:
        print(f'Not found: {src_path} (skipping)')

---
## Summary — What to do with the downloaded files

1. Place the `.pth` files in the **`models/`** folder of your local project:
   ```
   cyberbulling-detection/
   └── models/
       ├── best_model_mbert.pth
       ├── best_model_muril.pth
       ├── final_model_mbert.pth
       └── final_model_muril.pth
   ```

2. **Run local evaluation** (after pointing `config.py` to the right model):
   ```bash
   python -m src.evaluate
   python -m src.compare_models
   python -m src.baselines
   ```

3. **Update notebooks 03 and 04** with the actual training metrics and predictions.

> Your model checkpoints are also saved permanently in Google Drive at `My Drive/cyberbullying_models/`

---
## Optional — Run Baselines (SVM + LSTM) on Colab

If you want to run the baseline comparison here instead of locally:

In [ ]:
# Optional — only run if you want baseline results here
# !python -m src.baselines
# !python -m src.utils.fleiss_kappa
print('Uncomment the lines above to run baselines and inter-annotator agreement.')